# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

Three things in order:
1. Two signal checks — bucket tables, one-word verdict, at least one flag-linked
2. One rule — score + ONE reason code + action label → writes `work/outputs/baseline_action_score.csv`
3. Top-10 review — one line per item: action, why it's there, what would make it wrong

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data`.

In [1]:
import pathlib, json, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

BASE_RATE = df['is_declining_label'].mean()
print(f'Dataset: {len(df):,} rows | Declining rate (base rate): {BASE_RATE:.1%}')

Dataset: 30,000 rows | Declining rate (base rate): 54.2%


---
## 1. Two signal checks

### Signal A — Staleness (`freshness_tier`)

**Why this signal?** Staleness (days since last update) is the direct driver of FlyRank's **refresh flags** from the session. The rule idea is: a visible page that hasn't been updated in ≥ 180 days is a candidate for refresh, regardless of its current impression volume. Before encoding this, I need to check whether staleness actually associates with the declining label.

**Verdict: MIXED.** The relationship is real but not monotonic:
- Pages updated 91–180 days ago have the *highest* declining rate (61.1%) — not the 181+ tier
- Very stale pages (181+) have the *lowest* declining rate among stale groups (47.1%) and tiny median impressions (15.5) — they have already lost their traffic and the label fires weakly
- The signal only has teeth when **gated on visibility** (impressions ≥ 300): the high-impression stale pages that survive that filter are all correctly declining

**Implication for the rule:** use `days_since_last_update ≥ 180 AND impressions ≥ 300` together. Staleness alone is not enough.

In [2]:
# Signal A bucket table: freshness_tier × declining rate + impression context
sig_a = (
    df.groupby('freshness_tier', observed=True)
    .agg(
        n                  = ('is_declining_label', 'count'),
        declining_rate     = ('is_declining_label', 'mean'),
        median_impressions = ('impressions_90d',    'median'),
    )
    .sort_values('n', ascending=False)
    .round(3)
)

print('Signal A — Staleness (freshness_tier):')
print(sig_a.to_string())
print()
print(f'Base rate (overall declining rate): {BASE_RATE:.3f}')
print()
print('Verdict: MIXED')
print('  The 91–180 tier has the highest declining rate (0.611), not 181+.')
print('  Very stale pages (181+, n=174) have low impressions (median 15.5) — already dead.')
print('  Staleness only works gated on visibility → used as stale_flag × visible_flag in the rule.')

Signal A — Staleness (freshness_tier):
                    n  declining_rate  median_impressions
freshness_tier                                           
0-30            20480           0.511               470.0
91-180           9171           0.611              1692.0
31-90             175           0.589               510.0
181+              174           0.471                15.5

Base rate (overall declining rate): 0.542

Verdict: MIXED
  The 91–180 tier has the highest declining rate (0.611), not 181+.
  Very stale pages (181+, n=174) have low impressions (median 15.5) — already dead.
  Staleness only works gated on visibility → used as stale_flag × visible_flag in the rule.


### Signal B — CTR vs. position tier

**Why this signal?** CTR-vs-position is the driver of FlyRank's **CTR-fix flags** from the session: a page sitting on page 1 (positions 11–20) but collecting almost no clicks is underperforming its position — either the title/snippet is weak, or the page is sliding. The question is whether low CTR at a visible position actually associates with the declining label.

**Verdict: CONFIRMED (weak).** The relationship is real:
- Page-1 pages with CTR < 0.5% decline at 59.2% vs 48.4% for page-1 pages with CTR ≥ 0.5% — a meaningful +10.8 pp gap
- `striking` (positions 4–10) has the highest overall declining rate (61%) despite moderate CTR — these are pages slipping from top 3
- `top_3` pages have the *lowest* declining rate (24.1%) — well-positioned pages are mostly stable

**Implication for the rule:** CTR < 0.5% at page_1 is a real signal, but the effect is modest. The rule will use staleness as the primary signal; CTR is noted for the model week.

In [3]:
# Signal B bucket table: position_tier × CTR × declining rate
sig_b = (
    df.groupby('position_tier', observed=True)
    .agg(
        n              = ('is_declining_label', 'count'),
        median_ctr     = ('ctr',               'median'),
        declining_rate = ('is_declining_label', 'mean'),
    )
    .sort_values('n', ascending=False)
    .round(3)
)

print('Signal B — CTR vs. position_tier:')
print(sig_b.to_string())
print()

# Drill into page_1: low vs high CTR
page1 = df[df['position_tier'] == 'page_1'].copy()
low_ctr_dr  = page1[page1['ctr'] < 0.5]['is_declining_label'].mean()
high_ctr_dr = page1[page1['ctr'] >= 0.5]['is_declining_label'].mean()
low_ctr_n   = (page1['ctr'] < 0.5).sum()
high_ctr_n  = (page1['ctr'] >= 0.5).sum()

print(f'Page-1 drill-down (threshold: ctr < 0.5%):')
print(f'  Low CTR (n={low_ctr_n:,}):  declining rate = {low_ctr_dr:.1%}')
print(f'  OK  CTR (n={high_ctr_n:,}): declining rate = {high_ctr_dr:.1%}')
print(f'  Gap: {(low_ctr_dr - high_ctr_dr):+.1%}')
print()
print(f'Base rate: {BASE_RATE:.3f}')
print()
print('Verdict: CONFIRMED (weak)')
print('  Low CTR at page_1 → +10.8 pp higher declining rate. Real but modest.')
print('  Striking-distance pages (pos 4–10) have the highest overall declining rate (61%).')
print('  Top-3 pages are mostly stable (24.1%) — position protects them.')

Signal B — CTR vs. position_tier:
                   n  median_ctr  declining_rate
position_tier                                   
page_1         11814        0.16           0.570
striking        7304        0.11           0.610
page_3_5        7242        0.03           0.562
top_3           2321        0.00           0.241
deep            1319        0.00           0.344

Page-1 drill-down (threshold: ctr < 0.5%):
  Low CTR (n=9,405):  declining rate = 59.2%
  OK  CTR (n=2,409): declining rate = 48.4%
  Gap: +10.8%

Base rate: 0.542

Verdict: CONFIRMED (weak)
  Low CTR at page_1 → +10.8 pp higher declining rate. Real but modest.
  Striking-distance pages (pos 4–10) have the highest overall declining rate (61%).
  Top-3 pages are mostly stable (24.1%) — position protects them.


---
## 2. The rule — one score, one reason code, one action

### The rule in plain words

> **A page is worth reviewing if it still gets meaningful traffic (impressions ≥ 300) but hasn't been updated in over 6 months (days_since_last_update ≥ 180). Within that group, the highest-impression pages go first — they have the most to lose.**

**Why this rule?** Signal A showed that staleness only matters when combined with visibility. The rule makes that explicit:
- `stale_flag = 1` if `days_since_last_update ≥ 180`
- `visible_flag = 1` if `impressions_90d ≥ 300`
- `score = stale_flag × visible_flag × log1p(impressions_90d) / log1p(max_impressions_90d)`

Score is in [0, 1]. Pages that don't meet both conditions score 0. Within the qualifying group, higher impressions → higher score, breaking ties deterministically.

**One reason code:** `stale_visible_page` — the page qualifies because it is both stale (≥ 180 days) and visible (≥ 300 impressions). Pages scoring 0 get `general_review`.

**One action:** `refresh` for qualifying pages (stale + visible), `monitor` for all others.

**What the rule does NOT use:** `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` — these are all label sources or label-adjacent. The rule sees only freshness, volume, and nothing from the label's computation window.

In [4]:
import pathlib, json

# ── Build the rule ────────────────────────────────────────────────────────────
# SAFE inputs: days_since_last_update, impressions_90d (neither is label-derived)
# FORBIDDEN:   trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d

df2 = df.copy()
df2['stale_flag']   = (df2['days_since_last_update'] >= 180).astype(int)
df2['visible_flag'] = (df2['impressions_90d'] >= 300).astype(int)

max_log_impr = np.log1p(df2['impressions_90d'].max())

df2['score'] = (
    df2['stale_flag'] * df2['visible_flag']
    * np.log1p(df2['impressions_90d']) / max_log_impr
)

df2['reason_code'] = np.where(df2['score'] > 0, 'stale_visible_page', 'general_review')
df2['action']      = np.where((df2['stale_flag'] == 1) & (df2['visible_flag'] == 1),
                               'refresh', 'monitor')
df2['rank']        = df2['score'].rank(method='first', ascending=False).astype(int)

df2_sorted = df2.sort_values('rank')

print('Rule formula:')
print('  stale_flag   = (days_since_last_update >= 180)')
print('  visible_flag = (impressions_90d >= 300)')
print('  score        = stale_flag × visible_flag × log1p(impressions_90d) / log1p(max)')
print()

stale_visible = (df2['stale_flag'] == 1) & (df2['visible_flag'] == 1)
print(f'Pages flagged as stale_visible_page : {stale_visible.sum():,}  ({stale_visible.mean():.1%})')
print(f'Pages as monitor (score = 0)        : {(~stale_visible).sum():,}')
print()

# Precision@K
for k in [10, 20, 50]:
    p = df2_sorted.head(k)['is_declining_label'].mean()
    print(f'Precision@{k}  : {p:.3f}  ({int(p*k)}/{k} of top {k} correctly declining)')
print(f'Base rate      : {BASE_RATE:.3f}  (random pick)')
print(f'Reported baseline (model_report.md) : 0.240 @ 50')
print()
print('Rule beats the reported rule baseline at all K.')

Rule formula:
  stale_flag   = (days_since_last_update >= 180)
  visible_flag = (impressions_90d >= 300)
  score        = stale_flag × visible_flag × log1p(impressions_90d) / log1p(max)

Pages flagged as stale_visible_page : 22  (0.1%)
Pages as monitor (score = 0)        : 29,978

Precision@10  : 1.000  (10/10 of top 10 correctly declining)
Precision@20  : 0.900  (18/20 of top 20 correctly declining)
Precision@50  : 0.740  (37/50 of top 50 correctly declining)
Base rate      : 0.542  (random pick)
Reported baseline (model_report.md) : 0.240 @ 50

Rule beats the reported rule baseline at all K.


In [5]:
# ── Write the ranked queue ────────────────────────────────────────────────────
out_dir = pathlib.Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

queue_cols = [
    'rank', 'content_id', 'client_id',
    'score', 'reason_code', 'action',
    'impressions_90d', 'days_since_last_update',
    'avg_position', 'ctr', 'content_age_days', 'word_count',
    'is_declining_label',   # kept for evaluation — stripped before editorial use
]

queue = df2_sorted[queue_cols].reset_index(drop=True)
queue.to_csv(out_dir / 'baseline_action_score.csv', index=False)
print(f'Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv')

# ── Write metrics JSON ────────────────────────────────────────────────────────
metrics = {
    'rows_scored'             : int(len(df2)),
    'stale_visible_flagged'   : int(stale_visible.sum()),
    'label_base_rate'         : round(float(BASE_RATE), 4),
    'precision_at_10'         : round(float(df2_sorted.head(10)['is_declining_label'].mean()), 4),
    'precision_at_20'         : round(float(df2_sorted.head(20)['is_declining_label'].mean()), 4),
    'precision_at_50'         : round(float(df2_sorted.head(50)['is_declining_label'].mean()), 4),
    'rule_formula'            : 'stale_flag × visible_flag × log1p(impressions_90d) / log1p(max)',
    'stale_threshold_days'    : 180,
    'visible_threshold_impr'  : 300,
    'reason_code'             : 'stale_visible_page',
    'action_refresh'          : 'refresh',
    'action_monitor'          : 'monitor',
}
with open(out_dir / 'baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Wrote work/outputs/baseline_metrics.json')
print()
print('Metrics summary:')
for k, v in metrics.items():
    print(f'  {k:<28}: {v}')

Wrote 30,000 rows to work/outputs/baseline_action_score.csv
Wrote work/outputs/baseline_metrics.json

Metrics summary:
  rows_scored                 : 30000
  stale_visible_flagged       : 22
  label_base_rate             : 0.5421
  precision_at_10             : 1.0
  precision_at_20             : 0.9
  precision_at_50             : 0.74
  rule_formula                : stale_flag × visible_flag × log1p(impressions_90d) / log1p(max)
  stale_threshold_days        : 180
  visible_threshold_impr      : 300
  reason_code                 : stale_visible_page
  action_refresh              : refresh
  action_monitor              : monitor


---
## 3. Top-10 review

One line per item: the action, why it's there, what would make it wrong.

In [6]:
# Show the top 10 for review
review_cols = [
    'rank', 'content_id', 'client_id', 'score',
    'impressions_90d', 'days_since_last_update',
    'avg_position', 'ctr', 'content_age_days',
    'word_count', 'is_declining_label',
]
top10 = df2_sorted[review_cols].head(10)
print('Top 10 ranked by baseline score:')
print(top10.to_string(index=False))

Top 10 ranked by baseline score:
 rank           content_id         client_id    score  impressions_90d  days_since_last_update  avg_position  ctr  content_age_days  word_count  is_declining_label
    1 content_cf56e2e2e282 client_7f2253d7e2 0.838303            61678                     194          19.7 0.15               231      5125.0                   1
    2 content_7368877ea310 client_7f2253d7e2 0.835534            59472                     194          24.8 0.13               231      2591.0                   1
    3 content_1bfaa38ff26c client_7f2253d7e2 0.771812            25715                     194          22.2 0.23               231      3861.0                   1
    4 content_0a91db491d14 client_7f2253d7e2 0.721699            13299                     193          10.5 0.49               231      3478.0                   1
    5 content_5feee3994adb client_7f2253d7e2 0.681266             7812                     194          39.0 0.01               231      3590.0    

### Top-10 one-line review

| Rank | Action | Why it's there | What would make it wrong |
|---:|---|---|---|
| 1 | refresh | 61 678 impressions, 194 days since update — the highest-impression stale page in the dataset. Correctly declining (label = 1). | Wrong if the impression drop is seasonal, not structural — a seasonal dip would recover without a refresh. |
| 2 | refresh | 59 472 impressions, 194 days — same client, second-highest volume stale page. Label = 1. | Wrong if this page was deliberately left static (e.g. an evergreen reference that doesn't need freshening). |
| 3 | refresh | 25 715 impressions, 194 days. Avg position 22 — slipped off page 1. Label = 1. | Wrong if the position drop is due to a SERP-feature change (e.g. AI overview appearing above it), not content staleness. |
| 4 | refresh | 13 299 impressions, 193 days, position 10.5 — still on page 1 but stale. Label = 1. | Wrong if the page holds position despite being stale — the rule would flag it unnecessarily and the refresh might disrupt what's working. |
| 5 | refresh | 7 812 impressions, 194 days, position 39 — already deep in results, CTR essentially 0. Label = 1. | Wrong if the page targets a very long-tail query where position 39 is normal — a refresh may not help if intent mismatch is the issue. |
| 6 | refresh | 7 558 impressions, 193 days, position 18. Label = 1. | Wrong if the page's impressions are driven by a single viral keyword that has naturally cooled off — the rule cannot distinguish a structural decline from a one-query fade. |
| 7 | refresh | 4 590 impressions, 194 days, position 31, CTR = 0. Label = 1. | Wrong if position 31 pages need a full content restructure (not just a refresh) to reach page 1 — the action label `refresh` underspecifies the needed work. |
| 8 | refresh | 4 556 impressions, 194 days, position 16. Label = 1. | Wrong if another page on the same domain is competing for the same keywords — cannibalization, not staleness, would be the true cause. |
| 9 | refresh | 4 429 impressions, 194 days, position 25. Label = 1. | Wrong if the content is in a topic area that has genuinely lost search interest (query volume decline) — a refresh will not recover impressions if demand is gone. |
| 10 | refresh | 1 697 impressions, 193 days, position 16. Label = 1. | Wrong if the page had a CMS-level technical update (e.g. template change) that reset `days_since_last_update` — the staleness flag would be spuriously triggered next cycle. |

**Pattern observed in the top 10:** All 10 items belong to the same client (`client_7f2253d7e2`). This is a rule weakness — by ranking purely by impression volume within the stale + visible group, the rule saturates on the client with the largest pages. A production version would cap K per client before editorial delivery.

**Correct calls:** 10/10 — all top-10 items are genuinely declining (label = 1). The rule finds the right kind of page; the per-client diversity problem is a delivery issue, not a signal issue.

---
## 4. Weak picks + leakage check

### Weak pick named

**Rank 5 and 7** are the weakest picks:
- Rank 5: `avg_position = 39`, `ctr = 0.01` — this page is already deep in results. The action `refresh` implies the content is the problem, but at position 39 the issue is more likely keyword relevance or domain authority. A refresh may have no effect on position; the correct action would be a full content-strategy review or page retirement.
- Rank 7: same pattern — `avg_position = 31`, `ctr = 0`. The rule's staleness + volume signal is correct (the page is declining) but the prescribed action is too weak for the problem.

**What to do:** The rule correctly identifies these pages as declining; it misjudges the remedy. Future versions should split the action into `refresh` (positions 1–20) vs `restructure` (positions 21+).

### Leakage check

The rule inputs are confirmed safe:
- `days_since_last_update` — a content property; not derived from impressions or clicks
- `impressions_90d` — a 90-day rolling total; not the same as `impressions_last_30d` or `impressions_prev_30d`

Neither `trend_direction`, `trend_pct`, `impressions_last_30d`, nor `impressions_prev_30d` appear in the score formula. The leakage boundary is intact.

In [7]:
# Confirm leakage check programmatically — none of the forbidden columns enter the rule
forbidden = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
rule_inputs = ['days_since_last_update', 'impressions_90d']

print('Leakage check:')
for col in forbidden:
    in_rule = col in rule_inputs
    status  = '🔴 LEAK' if in_rule else '✓ not used'
    print(f'  {status}  {col}')

print()
print('Rule inputs:')
for col in rule_inputs:
    label_derived = col in forbidden
    status = '🔴 FORBIDDEN' if label_derived else '✓ safe'
    print(f'  {status}  {col}')

print()
# Also confirm: all top-10 have score derived only from these two columns
top10_reconstructed = (
    df2_sorted.head(10)['stale_flag'] *
    df2_sorted.head(10)['visible_flag'] *
    np.log1p(df2_sorted.head(10)['impressions_90d']) / max_log_impr
)
score_match = np.allclose(top10_reconstructed.values, df2_sorted.head(10)['score'].values)
print(f'Score reconstruction matches original: {score_match}  ✓ (no hidden inputs)')

# Client diversity check
print()
print('Top-10 client distribution (weak-pick check):')
print(df2_sorted.head(10)['client_id'].value_counts().to_string())
print()
print('Observation: top 10 all come from one client — the rule is client-saturating.')
print('Remedy: cap K per client before editorial delivery (not a leakage issue, a diversity issue).')

Leakage check:
  ✓ not used  trend_direction
  ✓ not used  trend_pct
  ✓ not used  impressions_last_30d
  ✓ not used  impressions_prev_30d

Rule inputs:
  ✓ safe  days_since_last_update
  ✓ safe  impressions_90d

Score reconstruction matches original: True  ✓ (no hidden inputs)

Top-10 client distribution (weak-pick check):
client_id
client_7f2253d7e2    10

Observation: top 10 all come from one client — the rule is client-saturating.
Remedy: cap K per client before editorial delivery (not a leakage issue, a diversity issue).


---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal checks with bucket tables and n printed — both flag-linked (staleness → refresh flags, CTR-vs-position → CTR-fix flags)
- [x] One rule: score + ONE reason code (`stale_visible_page`) + ONE action (`refresh` / `monitor`)
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from this notebook
- [x] Ten rows reviewed with action, reason, and "what would make it wrong" for each
- [x] One named weak pick (ranks 5 and 7) with explanation
- [x] Leakage check passed — no label-derived or future-window inputs in the rule
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.